# MAE from scratch — STL-10 et défauts NEU

In [ ]:
# imports
import numpy as np
import matplotlib.pyplot as plt
from scripts.datasetdwnld import read_all_images
DATA_PATH = "/Users/vitt/school/introduction-to-computer-vision/project/mae/data/stl10_binary.tar.gz"

sys.version_info(major=3, minor=14, micro=5, releaselevel='final', serial=0)


In [ ]:
def patchify(imgs,P=16):
    B, H, W, C = imgs.shape
    N = H*W//P**2
    patch_dim = (P**2)*C
    patches = np.zeros((B,N,P,P,C))
    for i in range(B):
        for r in range(H//P):
            for c in range(W//P):
                patches[i][r*(W//P)+c] = imgs[i][r*P:r*P+P,c*P:c*P+P,:]
    patches = patches.reshape(B, N, patch_dim)
    return patches
def unpatchify(patches, P=16):
    B, N, patch_dim = patches.shape
    C = patch_dim//P**2
    H = int(N**0.5) * P
    W = H
    imgs = np.zeros((B, H, W, C))
    patches = patches.reshape(B,N,P,P,C)
    for i in range(B):
        for r in range(H//P):
            for c in range(W//P):
                imgs[i][r*P:r*P+P,c*P:c*P+P,:] = patches[i][r*(W//P)+c]
    return imgs

In [ ]:
class PatchEmbed:
    def __init__(self, patch_dim, D):
        self.W = np.random.randn(patch_dim, D)*np.sqrt(2.0/patch_dim)
        self.b = np.zeros(D)
        self.cache = None
    def forward(self, patches):
        self.cache = patches
        Y = patches @ self.W + self.b
        return Y
    def backward(self, dY):
        self.db = dY.sum(axis=(0,1))
        dX = dY @ self.W.T
        self.dW = self.cache.reshape(-1, self.cache.shape[-1]).T @ dY.reshape(-1, dY.shape[-1])
        return dX


In [ ]:
def pos_embed_1d(positions, d):
    k = np.arange(d // 2)
    denom = 10000 ** (2 * k / d)
    args = positions[:,None] / denom[None,:]
    pe = np.zeros((len(positions), d))
    pe[:,0::2] = np.sin(args)
    pe[:,1::2] = np.cos(args)
    return pe
def pos_embed_2d(N, D):
    grid = int(N**0.5)
    i = np.arange(N)
    lignes = i // grid
    cols = i % grid
    pe_lignes = pos_embed_1d(lignes, D // 2)
    pe_cols = pos_embed_1d(cols, D // 2)
    return np.concatenate([pe_lignes, pe_cols], axis=1)


In [ ]:
#mask
def random_masking(x, mask_ratio):
    B, N, D = x.shape
    N_vis = int(N*(1-mask_ratio))
    noise = np.random.rand(B, N)
    ids_shuffle = np.argsort(noise, axis=1)
    ids_restore = np.argsort(ids_shuffle, axis=1)
    ids_keep = ids_shuffle[:, :N_vis]
    x_vis = np.take_along_axis(x, ids_keep[:, :, None].repeat(D, axis=2), axis=1)
    mask_shuffled = np.ones((B, N))
    mask_shuffled[:, :N_vis] = 0 
    mask = np.take_along_axis(mask_shuffled, ids_restore, axis=1)
    return x_vis, mask, ids_restore


In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True) #Stabilité si x tres grand
    return np.exp(x)/np.sum(np.exp(x), axis=axis, keepdims=True)

def gelu(x):
    return (1/2)*x*(1+np.tanh(np.sqrt(2.0/(np.pi))*(x+0.044715*(x**3))))

def gelu_backward(x):
    K = np.sqrt(2.0 / np.pi)
    u = K * (x + 0.044715 * x**3)
    t = np.tanh(u)
    du = K * (1.0 + 0.134145 * x**2)
    return 0.5 * (1.0 + t) + 0.5 * x * (1.0 - t**2) * du



In [ ]:
#tests

layer = PatchEmbed(768, 192)
X = np.random.randn(2, 36, 768)

Y = layer.forward(X)
dY = 2 * Y
layer.backward(dY)
eps = 1e-5
layer.W[0][0] += eps
L_plus = (layer.forward(X)**2).sum()
layer.W[0][0] -= 2*eps
L_moins = (layer.forward(X)**2).sum()
layer.W[0][0] += eps

grad_num = (L_plus - L_moins)/(2*eps)
print(grad_num, layer.dW[0][0])
err = abs(grad_num - layer.dW[0][0]) / abs(grad_num)
print(err)  

6.193213994265533 6.193214104636574
1.7821286540005563e-08


## Étapes 4 à 12 — couches, modèle, tests

Le modèle est dans `mae.py` : torch sert seulement de backend de tableaux, tout le forward/backward est écrit à la main. On teste sur CPU en double précision (le gradient check a besoin de précision).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
import mae as M, train as T
M.use("cpu", torch.float64)

In [ ]:
# Étape 4 : LayerNorm (moyenne 0 / variance 1 par token) et softmax stable
x = torch.randn(3, 5, 8, dtype=torch.float64) * 4 + 2
y = M.LayerNorm(8).forward(x)
print("layernorm  mean=%.1e  var=%.3f" % (y.mean(), y.var(unbiased=False)))
s = M.softmax(torch.tensor([[1000., 1001., 1002.]], dtype=torch.float64))
print("softmax stable :", s.numpy(), "somme =", float(s.sum()))

In [ ]:
# Étape 5 : multi-head attention. Sortie (B,N,D), poids d'attention somment à 1 par requête
mha = M.MHA(12, 3)
out = mha.forward(torch.randn(2, 5, 12, dtype=torch.float64))
_, _, _, P, _ = mha.cache
print("sortie", tuple(out.shape), "| somme attention par requête ~", round(float(P.sum(-1).mean()), 4))

In [ ]:
# Étape 6 : bloc pré-norm, forward stable
blk = M.Block(12, 3)
print("bloc fini (pas de nan) :", bool(torch.isfinite(blk.forward(torch.randn(2, 5, 12, dtype=torch.float64))).all()))

In [ ]:
# Étapes 7-8 : encodeur (visibles seulement) et décodeur (séquence complète)
cfg = M.default_config()
enc = M.Encoder(cfg); imgs = torch.rand(2, 96, 96, 3, dtype=torch.float64)
latent, mask, ids = enc.forward(imgs, 0.75)
pred = M.Decoder(cfg).forward(latent, ids)
print("latent", tuple(latent.shape), "| pred", tuple(pred.shape))

In [ ]:
# Étape 9 : MSE sur les patchs masqués, cible normalisée par patch
loss, _ = M.mae_loss_and_grad(imgs, pred, mask)
tgt = M.patchify(imgs); mu, std = M.patch_stats(imgs)
loss0, _ = M.mae_loss_and_grad(imgs, (tgt - mu) / std, mask)
print("loss =", round(float(loss), 3), ">= 0   |   loss(pred=cible) =", round(float(loss0), 6))

In [ ]:
# Preuve : gradient check global (différences finies vs backward manuel), CPU float64
import random
small = dict(img=32, P=16, C=3, D=16, enc_depth=2, enc_heads=2,
             dec_dim=8, dec_depth=1, dec_heads=2, mlp_ratio=2, mask_ratio=0.5, norm_pix=True)
m = M.MAE(small); xi = torch.rand(2, 32, 32, 3, dtype=torch.float64)
def Lf():
    torch.manual_seed(0); return m.forward(xi)[0].item()
torch.manual_seed(0); m.forward(xi); m.backward()
worst = 0.0; random.seed(1)
for o, n in m.params():
    p = getattr(o, n).view(-1); dp = getattr(o, "d" + n).reshape(-1)
    for _ in range(3):
        i = random.randrange(p.numel()); old = p[i].item()
        p[i] = old + 1e-6; Lp = Lf(); p[i] = old - 1e-6; Lm = Lf(); p[i] = old
        num = (Lp - Lm) / 2e-6; ana = dp[i].item(); d = abs(num) + abs(ana)
        worst = max(worst, abs(num - ana) if d < 1e-6 else abs(num - ana) / d)
print("pire erreur relative globale =", f"{worst:.1e}",
      "-> backprop correcte" if worst < 1e-4 else "PROBLEME")

In [ ]:
# Étape 10 : sur-apprentissage d'un seul batch. La perte doit chuter
xb, _ = T.load_stl("unlabeled", 16)
h = T.overfit_one_batch(dict(small, img=96), xb, steps=150, lr=1e-3)
plt.figure(figsize=(5, 3)); plt.plot(h)
plt.xlabel("itération"); plt.ylabel("perte"); plt.title("overfit d'un batch")
plt.tight_layout(); plt.show()
print("loss %.2f -> %.3f" % (h[0], h[-1]))

## Entraînement complet et ablation

Le pré-entraînement complet (GPU) est lancé par `python experiments.py` puis `python experiments_neu.py`. On recharge les résultats sauvegardés.

In [ ]:
res = json.load(open("results/ablation.json"))
lc = np.load("results/loss_curves.npz")
plt.figure(figsize=(6, 4))
for r in res["ratios"]:
    plt.plot(lc[str(r)], label=f"mask {r}")
plt.xlabel("epoch"); plt.ylabel("perte MSE (patchs masqués)")
plt.title("Pretraining STL-10"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Reconstructions STL : original | masqué | reconstruit
rec = np.load("results/recon.npz"); o, mk, re = rec["orig"], rec["masked"], rec["recon"]
fig, ax = plt.subplots(3, 6, figsize=(11, 5.5))
for j in range(6):
    for i, (im, lab) in enumerate([(o, "original"), (mk, "masqué 75%"), (re, "reconstruit")]):
        ax[i, j].imshow(im[j]); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        if j == 0: ax[i, 0].set_ylabel(lab)
plt.tight_layout(); plt.show()

In [ ]:
# Sonde linéaire par taux de masquage
ratios = sorted(res["results"], key=float)
accs = [res["results"][r]["acc"] for r in ratios]; base = res["baseline_acc"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar([f"mask {r}" for r in ratios] + ["aléatoire", "hasard"], accs + [base, 0.1])
ax[0].set_ylabel("exactitude"); ax[0].set_title("Sonde vs références"); ax[0].tick_params(axis="x", rotation=45)
ax[1].plot([float(r) for r in ratios], accs, "o-"); ax[1].axhline(base, ls="--", color="gray", label="aléatoire")
ax[1].set_xlabel("taux de masquage"); ax[1].set_ylabel("exactitude"); ax[1].set_title("Ablation"); ax[1].legend()
plt.tight_layout(); plt.show()
print("baseline %.3f | " % base + "  ".join(f"{r}:{res['results'][r]['acc']:.3f}" for r in ratios))

## Cas applicatif : défauts de surface d'acier (NEU)

In [ ]:
# aperçu : un exemple par classe de défaut
Xn, yn, _, _, cls = T.load_neu()
fig, ax = plt.subplots(1, 6, figsize=(12, 2.3))
for c in range(6):
    i = np.where(yn == c)[0][0]
    ax[c].imshow(Xn[i], cmap="gray"); ax[c].axis("off"); ax[c].set_title(cls[c], fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
neu = json.load(open("results/neu.json"))
print("MAE %.3f  vs encodeur aléatoire %.3f" % (neu["acc_full"], neu["acc_base"]))
pc = neu["per_class"]; tot = [k * 6 for k in pc]
plt.figure(figsize=(6, 4))
plt.plot(tot, [neu["mae_eff"][str(k)] for k in pc], "o-", label="MAE pré-entraîné")
plt.plot(tot, [neu["rnd_eff"][str(k)] for k in pc], "s--", color="gray", label="encodeur aléatoire")
plt.xlabel("images étiquetées"); plt.ylabel("exactitude (6 défauts)")
plt.title("NEU — efficacité en annotations"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
rec = np.load("results/neu_recon.npz"); o, mk, re = rec["orig"], rec["masked"], rec["recon"]
fig, ax = plt.subplots(3, 6, figsize=(11, 5.5))
for j in range(6):
    for i, (im, lab) in enumerate([(o, "original"), (mk, "masqué 75%"), (re, "reconstruit")]):
        ax[i, j].imshow(im[j], cmap="gray"); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        if j == 0: ax[i, 0].set_ylabel(lab)
plt.tight_layout(); plt.show()